# EV Charging Gaps – Ruhr Area

Where are public charging points missing? A **Poisson regression** estimates the expected number of public charging points in each 1 km grid cell of nine Ruhr cities from population, home ownership, age and shops. Cells with clearly fewer charging points than expected are **charging gaps**.

**Data:** Ladesäulenregister (Bundesnetzagentur) · Census 2022 1 km grid (Destatis) · shops and city boundaries (OpenStreetMap)

In [ ]:
# 1. Install packages
!pip install -q osmnx contextily

In [ ]:
# 2. Imports and settings
import io, re, zipfile, requests, warnings, os
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from shapely.geometry import box
import osmnx as ox, contextily as ctx
import statsmodels.formula.api as smf
import statsmodels.api as sm
warnings.filterwarnings("ignore")
os.makedirs("images", exist_ok=True); os.makedirs("results", exist_ok=True)

CITIES = ["Duisburg", "Essen", "Mülheim an der Ruhr", "Oberhausen", "Gelsenkirchen",
          "Bochum", "Dortmund", "Herne", "Witten"]
ZBASE = "https://www.destatis.de/static/DE/zensus/gitterdaten/"
ZVARS = {
    "einwohner": "Zensus2022_Bevoelkerungszahl.zip",
    "eigentum":  "Eigentuemerquote_in_Gitterzellen.zip",
    "alter":     "Durchschnittsalter_in_Gitterzellen.zip",
}
BNETZA_PAGE = "https://www.bundesnetzagentur.de/DE/Fachthemen/ElektrizitaetundGas/E-Mobilitaet/DownloadundKontakt.html"
BNETZA_CSV = "https://data.bundesnetzagentur.de/Bundesnetzagentur/DE/Fachthemen/ElektrizitaetundGas/E-Mobilitaet/Ladesaeulenregister_BNetzA_2026-09-01.csv"

In [ ]:
# 3. City boundaries
cities = pd.concat([ox.geocode_to_gdf(f"{c}, Germany").assign(stadt=c) for c in CITIES])
cities = cities[["stadt", "geometry"]].to_crs(3035).reset_index(drop=True)
area = cities.union_all()
xmin, ymin, xmax, ymax = cities.total_bounds
cities.plot(column="stadt", edgecolor="black", figsize=(10, 5));

In [ ]:
# 4. Census 2022 – 1 km grid cells
def load_var(zipname):
    r = requests.get(ZBASE + zipname, timeout=600); r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    name = [n for n in z.namelist() if n.lower().endswith(".csv") and "1km" in n.lower()][0]
    raw = z.read(name)
    enc = "utf-8"
    try: raw[:200000].decode("utf-8")
    except UnicodeDecodeError: enc = "latin1"
    df = pd.read_csv(io.BytesIO(raw), sep=";", decimal=",", encoding=enc, low_memory=False)
    cols = {c.lower(): c for c in df.columns}
    xc = [cols[c] for c in cols if c.startswith("x_mp")][0]
    yc = [cols[c] for c in cols if c.startswith("y_mp")][0]
    df = df[(df[xc] >= xmin) & (df[xc] <= xmax) & (df[yc] >= ymin) & (df[yc] <= ymax)]
    val = [c for c in df.columns if c not in (xc, yc) and not c.lower().startswith("gitter_id")
           and "werterlaeuternde" not in c.lower()][0]
    out = df[[xc, yc, val]].copy(); out.columns = ["x", "y", "wert"]
    out["wert"] = pd.to_numeric(out["wert"].astype(str).str.replace(",", "."), errors="coerce")
    print(f"{zipname}: {len(out)} cells, column '{val}'")
    return out

grid = None
for key, zn in ZVARS.items():
    d = load_var(zn).rename(columns={"wert": key})
    grid = d if grid is None else grid.merge(d, on=["x", "y"], how="left")
grid = grid[grid["einwohner"] > 0].copy()
inside = gpd.GeoSeries(gpd.points_from_xy(grid.x, grid.y), crs=3035).within(area).values
grid = grid[inside].reset_index(drop=True)
print(len(grid), "populated 1 km cells,", int(grid.einwohner.sum()), "inhabitants")

In [ ]:
# 5. Public charging points – Ladesäulenregister (Bundesnetzagentur)
url = BNETZA_CSV
try:
    page = requests.get(BNETZA_PAGE, timeout=60).text
    links = re.findall(r'href="([^"]*Ladesaeulenregister[^"]*\.csv)', page)
    if links:
        url = links[0] if links[0].startswith("http") else "https://www.bundesnetzagentur.de" + links[0]
except Exception as e:
    print("page not read, using default URL:", e)
print("Download:", url)
raw = requests.get(url, timeout=600).content
for enc in ("utf-8-sig", "cp1252", "latin1"):
    try: text = raw.decode(enc); break
    except UnicodeDecodeError: pass
lines = text.splitlines()
hdr = [i for i, l in enumerate(lines[:50]) if "Breitengrad" in l][0]
lp = pd.read_csv(io.StringIO("\n".join(lines[hdr:])), sep=";", dtype=str, on_bad_lines="skip")
cl = {c.lower(): c for c in lp.columns}
lat = [cl[c] for c in cl if "breitengrad" in c][0]
lon = [cl[c] for c in cl if "ngengrad" in c][0]
anz = [cl[c] for c in cl if "anzahl" in c and "ladepunkte" in c][0]
num = lambda s: pd.to_numeric(s.astype(str).str.strip().str.replace(",", "."), errors="coerce")
lp = pd.DataFrame({"lat": num(lp[lat]), "lon": num(lp[lon]), "ladepunkte": num(lp[anz]).fillna(1)}).dropna()
lp = gpd.GeoDataFrame(lp, geometry=gpd.points_from_xy(lp.lon, lp.lat), crs=4326).to_crs(3035)
lp = lp[lp.within(area)].reset_index(drop=True)
print(len(lp), "charging stations,", int(lp.ladepunkte.sum()), "charging points in the 9 cities")

In [ ]:
# 6. Shops from OpenStreetMap (with retry)
import time
ox.settings.timeout = 300
shops = []
for c in CITIES:
    for attempt in range(5):
        try:
            s = ox.features_from_place(f"{c}, Germany", tags={"shop": True})
            break
        except Exception as e:
            print(c, "– retry", attempt + 1, type(e).__name__)
            time.sleep(30)
    shops.append(s[["geometry"]])
    print(c, len(s))
    time.sleep(10)
shops = pd.concat(shops)
shops = shops[~shops.index.duplicated()].to_crs(3035)
shops = gpd.GeoDataFrame(geometry=shops.geometry.representative_point(), crs=3035)
print(len(shops), "shops")

In [ ]:
# 7. Count charging points and shops per 1 km cell
def cell_xy(g):
    return (np.floor(g.x / 1000) * 1000 + 500).astype(int), (np.floor(g.y / 1000) * 1000 + 500).astype(int)

lp["x"], lp["y"] = cell_xy(lp.geometry)
shops["x"], shops["y"] = cell_xy(shops.geometry)
grid["x"], grid["y"] = grid.x.astype(int), grid.y.astype(int)
grid = grid.merge(lp.groupby(["x", "y"])["ladepunkte"].sum().reset_index(), on=["x", "y"], how="left")
grid = grid.merge(shops.groupby(["x", "y"]).size().rename("geschaefte").reset_index(), on=["x", "y"], how="left")
grid[["ladepunkte", "geschaefte"]] = grid[["ladepunkte", "geschaefte"]].fillna(0)
for c in ["eigentum", "alter"]:
    grid[c] = grid[c].fillna(grid[c].median())

cells = gpd.GeoDataFrame(grid, geometry=[box(x - 500, y - 500, x + 500, y + 500) for x, y in zip(grid.x, grid.y)], crs=3035)
cells = gpd.sjoin(cells, cities, predicate="intersects", how="left")
cells = cells[~cells.index.duplicated()].drop(columns="index_right")
print(len(cells), "cells,", int(cells.ladepunkte.sum()), "charging points assigned")
cells[["einwohner", "eigentum", "alter", "geschaefte", "ladepunkte"]].describe().round(1)

In [ ]:
# 8. Poisson regression: expected number of charging points per cell
cells["log_einwohner"] = np.log1p(cells["einwohner"])
cells["log_geschaefte"] = np.log1p(cells["geschaefte"])
model = smf.glm("ladepunkte ~ log_einwohner + log_geschaefte + eigentum + alter",
                data=cells, family=sm.families.Poisson()).fit()
print(model.summary())
pseudo_r2 = 1 - model.llf / model.llnull
print("McFadden pseudo R²:", round(pseudo_r2, 3))
coef = pd.DataFrame({"coef": model.params, "IRR": np.exp(model.params), "p": model.pvalues}).round(4)
display(coef)

cells["erwartet"] = model.predict(cells)
cells["differenz"] = cells["ladepunkte"] - cells["erwartet"]
bins = [-np.inf, -10, -3, 3, 10, np.inf]
names = ["Deutliche Lücke", "Leichte Lücke", "Ausgeglichen", "Leicht über Erwartung", "Deutlich über Erwartung"]
cells["klasse"] = pd.cut(cells["differenz"], bins=bins, labels=names).astype(str)
cells["klasse"].value_counts()

In [ ]:
# 9. Summary per city and largest gaps
gap = cells["klasse"] == "Deutliche Lücke"
city = cells.groupby("stadt").agg(einwohner=("einwohner", "sum"), ladepunkte=("ladepunkte", "sum"),
                                  zellen_luecke=("klasse", lambda s: (s == "Deutliche Lücke").sum()))
city["lp_je_1000_ew"] = (city["ladepunkte"] / city["einwohner"] * 1000).round(2)
city["ew_in_luecken"] = cells[gap].groupby("stadt")["einwohner"].sum()
city["ew_in_luecken"] = city["ew_in_luecken"].fillna(0).astype(int)
city["anteil_ew_in_luecken_%"] = (city["ew_in_luecken"] / city["einwohner"] * 100).round(1)
city = city.sort_values("lp_je_1000_ew")
display(city)
top = cells.sort_values("differenz")[["stadt", "einwohner", "geschaefte", "eigentum", "ladepunkte", "erwartet", "differenz"]].head(15).round(1)
display(top)
print("Total:", int(cells.einwohner.sum()), "inhabitants,", int(cells.ladepunkte.sum()), "charging points,",
      int(gap.sum()), "cells with a clear gap,", round(cells[gap].einwohner.sum() / cells.einwohner.sum() * 100, 1), "% of inhabitants")

In [ ]:
# 10. Maps
m = cells.to_crs(3857); cb = cities.to_crs(3857)

def draw_cities(ax):
    cb.boundary.plot(ax=ax, color="#333333", linewidth=0.8)
    for _, r in cb.iterrows():
        p = r.geometry.representative_point()
        ax.annotate(r["stadt"], (p.x, p.y), ha="center", fontsize=8, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7))

# Map 1: charging gaps
COL = {"Deutliche Lücke": "#b2182b", "Leichte Lücke": "#ef8a62", "Ausgeglichen": "#e0e0e0",
       "Leicht über Erwartung": "#67a9cf", "Deutlich über Erwartung": "#2166ac"}
fig, ax = plt.subplots(figsize=(14, 8))
for k, c in COL.items():
    sub = m[m.klasse == k]
    if len(sub): sub.plot(ax=ax, color=c, edgecolor="white", linewidth=0.3, alpha=0.85)
draw_cities(ax)
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldGrayCanvas, attribution_size=6)
ax.legend(handles=[Patch(color=c, label=k) for k, c in COL.items()],
          title="Ladepunkte: beobachtet − erwartet", loc="upper left", fontsize=9)
ax.set_title("Lücken bei öffentlichen Ladepunkten – Ruhrgebiet (1-km-Raster, Poisson-Regression)")
ax.set_axis_off()
plt.savefig("images/01_Ladeluecken_Ruhrgebiet.png", dpi=200, bbox_inches="tight"); plt.show()

# Map 2: observed charging points per cell
BINS = [(0, 0, "0", "#ffffcc"), (1, 5, "1–5", "#c7e9b4"), (6, 20, "6–20", "#7fcdbb"),
        (21, 50, "21–50", "#41b6c4"), (51, 100, "51–100", "#2c7fb8"), (101, 1e9, "über 100", "#253494")]
fig, ax = plt.subplots(figsize=(14, 8))
for lo, hi, lab, c in BINS:
    sub = m[(m.ladepunkte >= lo) & (m.ladepunkte <= hi)]
    if len(sub): sub.plot(ax=ax, color=c, edgecolor="white", linewidth=0.3, alpha=0.85)
draw_cities(ax)
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldGrayCanvas, attribution_size=6)
ax.legend(handles=[Patch(color=c, label=lab) for _, _, lab, c in BINS],
          title="Öffentliche Ladepunkte je km²", loc="upper left", fontsize=9)
ax.set_title("Öffentliche Ladepunkte – Ruhrgebiet (Ladesäulenregister, 1-km-Raster)")
ax.set_axis_off()
plt.savefig("images/02_Ladepunkte_Ruhrgebiet.png", dpi=200, bbox_inches="tight"); plt.show()

# Chart 3: observed vs expected
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(cells["erwartet"], cells["ladepunkte"], s=10, alpha=0.5, color="#2c7fb8")
lim = max(cells["erwartet"].max(), cells["ladepunkte"].max()) * 1.05
ax.plot([0, lim], [0, lim], color="black", linewidth=1)
ax.set_xlabel("Erwartete Ladepunkte (Modell)"); ax.set_ylabel("Beobachtete Ladepunkte")
ax.set_title(f"Beobachtet vs. erwartet (Pseudo-R² = {pseudo_r2:.2f})")
plt.savefig("images/03_Beobachtet_vs_Erwartet.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
# 11. Export results and download
out = cells.drop(columns=["log_einwohner", "log_geschaefte"]).round(2)
out.to_crs(25832).to_file("results/ladeluecken_ruhrgebiet.gpkg", driver="GPKG")
city.to_csv("results/ladeluecken_je_stadt.csv")
coef.to_csv("results/poisson_koeffizienten.csv")
!zip -q -r ev_charging_results.zip images results
try:
    from google.colab import files
    files.download("ev_charging_results.zip")
except ImportError:
    print("Results saved in ev_charging_results.zip")